In [ ]:
# 1. INPUT

from pathlib import Path

DATA_DIR = Path("datasets/text-files")

files = sorted(DATA_DIR.glob("book_*.txt"))

print("Number of input files:", len(files))

for file_path in files:
    print(file_path.name)

Number of input files: 5
book_01.txt
book_02.txt
book_03.txt
book_04.txt
book_05.txt


In [ ]:
# 2. MAP FUNCTION

import re

def mapper(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read().lower()

    words = re.findall(r"\b[a-z0-9']+\b", text)

    pairs = [(word, 1) for word in words]

    return file_path.name, pairs

In [ ]:
# 3. RUN MAPPERS IN PARALLEL

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=5) as executor:
    mapper_results = list(executor.map(mapper, files))

for filename, pairs in mapper_results:
    print(f"\n{filename}")
    print(pairs[:5])


book_01.txt
[('distributed', 1), ('systems', 1), ('process', 1), ('large', 1), ('volumes', 1)]

book_02.txt
[('big', 1), ('data', 1), ('pipelines', 1), ('often', 1), ('begin', 1)]

book_03.txt
[('cloud', 1), ('storage', 1), ('enables', 1), ('scalable', 1), ('processing', 1)]

book_04.txt
[('text', 1), ('analytics', 1), ('includes', 1), ('frequency', 1), ('analysis', 1)]

book_05.txt
[('performance', 1), ('tuning', 1), ('may', 1), ('involve', 1), ('custom', 1)]


In [ ]:
# 4A. COMBINE: local word counts per file

from collections import Counter

combined_results = []

for filename, pairs in mapper_results:
    local_counts = Counter(
        dict()
    )

    for word, count in pairs:
        local_counts[word] += count

    combined_pairs = list(local_counts.items())

    combined_results.append((filename, combined_pairs))

    print(f"\nCombiner for {filename}")
    print(combined_pairs[:10])


Combiner for book_01.txt
[('distributed', 1), ('systems', 1), ('process', 1), ('large', 1), ('volumes', 1), ('of', 2), ('information', 1), ('across', 1), ('many', 1), ('machines', 1)]

Combiner for book_02.txt
[('big', 1), ('data', 1), ('pipelines', 1), ('often', 1), ('begin', 1), ('with', 1), ('log', 1), ('ingestion', 1), ('and', 2), ('parsing', 1)]

Combiner for book_03.txt
[('cloud', 1), ('storage', 1), ('enables', 1), ('scalable', 1), ('processing', 1), ('for', 1), ('educational', 1), ('projects', 1), ('mapreduce', 1), ('supports', 1)]

Combiner for book_04.txt
[('text', 1), ('analytics', 1), ('includes', 1), ('frequency', 1), ('analysis', 1), ('keyword', 1), ('extraction', 1), ('and', 3), ('indexing', 1), ('an', 1)]

Combiner for book_05.txt
[('performance', 1), ('tuning', 1), ('may', 1), ('involve', 1), ('custom', 1), ('partitioners', 1), ('and', 2), ('combiners', 1), ('combiner', 1), ('logic', 1)]


In [ ]:
# 4B. PARTITION: send each word to one reducer

NUM_REDUCERS = 1

partitions = {
    reducer_id: []
    for reducer_id in range(NUM_REDUCERS)
}

for filename, combined_pairs in combined_results:
    for word, count in combined_pairs:
        reducer_id = hash(word) % NUM_REDUCERS

        partitions[reducer_id].append((word, count))

for reducer_id, pairs in partitions.items():
    print(f"\nPartition for Reducer {reducer_id}")
    print(pairs[:20])


Partition for Reducer 0
[('distributed', 1), ('systems', 1), ('process', 1), ('large', 1), ('volumes', 1), ('of', 2), ('information', 1), ('across', 1), ('many', 1), ('machines', 1), ('map', 1), ('tasks', 2), ('split', 1), ('the', 2), ('input', 1), ('and', 3), ('emit', 1), ('intermediate', 1), ('key', 2), ('value', 1)]


In [ ]:
# 5A. SHUFFLE & SORT: group values by word

from collections import defaultdict

shuffled_data = {}

for reducer_id, pairs in partitions.items():
    grouped_words = defaultdict(list)

    for word, count in pairs:
        grouped_words[word].append(count)

    shuffled_data[reducer_id] = dict(grouped_words)

for reducer_id, grouped_words in shuffled_data.items():
    print(f"\nShuffled data for Reducer {reducer_id}")

    for word, counts in list(grouped_words.items())[:10]:
        print(word, "->", counts)


Shuffled data for Reducer 0
distributed -> [1]
systems -> [1]
process -> [1]
large -> [1]
volumes -> [1]
of -> [2]
information -> [1]
across -> [1]
many -> [1, 1]
machines -> [1]


In [ ]:
# 5B. REDUCE: sum values for each word

reducer_results = {}

for reducer_id, grouped_words in shuffled_data.items():
    reduced_counts = {}

    for word, counts in grouped_words.items():
        reduced_counts[word] = sum(counts)

    reducer_results[reducer_id] = reduced_counts

for reducer_id, results in reducer_results.items():
    print(f"\nResults from Reducer {reducer_id}")

    for word, count in list(results.items())[:10]:
        print(word, "->", count)


Results from Reducer 0
distributed -> 1
systems -> 1
process -> 1
large -> 1
volumes -> 1
of -> 2
information -> 1
across -> 1
many -> 2
machines -> 1


In [ ]:
# OUTPUT: one file per reducer

from pathlib import Path

OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)

for reducer_id, results in reducer_results.items():
    output_file = OUTPUT_DIR / f"part-{reducer_id:05d}.txt"

    with open(output_file, "w", encoding="utf-8") as file:
        for word, count in sorted(results.items()):
            file.write(f"{word} {count}\n")

    print(f"Written: {output_file}")

Written: part-00000.txt
